# VisionGuard AI — Fine-tune EfficientNet-B4 on APTOS 2019

Run this notebook on **Google Colab with a free GPU** (Runtime → Change runtime type → GPU) to train the real Diabetic Retinopathy severity classifier that powers VisionGuard AI's production mode.

**Output:** `best_model.pth` — copy it to `backend/app/ml/weights/best_model.pth` and set `MODEL_MODE=production` in `backend/.env`.

This notebook mirrors `train_efficientnet_aptos.py` cell-by-cell if you'd rather run it as a script.

## 1. Install dependencies

In [ ]:
!pip install -q kaggle scikit-learn opencv-python-headless timm

## 2. Download the APTOS 2019 dataset

You need a free Kaggle account. Get your API token from kaggle.com/settings → "Create New Token", which downloads `kaggle.json`. Upload it below, or set the two env vars directly.

You'll also need to **accept the competition rules** at
https://www.kaggle.com/c/aptos2019-blindness-detection/rules before the download works.

In [ ]:
from google.colab import files
import os

uploaded = files.upload()  # select your kaggle.json
os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

In [ ]:
!kaggle competitions download -c aptos2019-blindness-detection -p data/
!unzip -q -o data/aptos2019-blindness-detection.zip -d data/aptos2019
!ls data/aptos2019

## 3. Imports & config

In [ ]:
import copy
import time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import cohen_kappa_score
from sklearn.model_selection import train_test_split
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset
import timm

DATA_DIR = Path('data/aptos2019')
OUTPUT_DIR = Path('checkpoints')
OUTPUT_DIR.mkdir(exist_ok=True)

NUM_CLASSES = 5
IMG_SIZE = 380
BATCH_SIZE = 16
EPOCHS = 20
LR = 3e-4
VAL_SPLIT = 0.15
SEED = 42
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

## 4. Dataset — with the same CLAHE preprocessing used in production

Training on exactly the distribution the inference API will see (center-crop → resize → CLAHE → ImageNet normalize) avoids a train/serve skew that would otherwise quietly hurt accuracy.

In [ ]:
def apply_clahe(image_rgb):
    lab = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    l = clahe.apply(l)
    lab = cv2.merge((l, a, b))
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)


class APTOSDataset(Dataset):
    def __init__(self, df, images_dir, img_size, train):
        self.df = df.reset_index(drop=True)
        self.images_dir = images_dir
        self.img_size = img_size
        self.train = train

        aug = [
            transforms.ToPILImage(),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.RandomRotation(20),
            transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ]
        plain = [
            transforms.ToPILImage(),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ]
        self.transform = transforms.Compose(aug if train else plain)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self.images_dir / f"{row['id_code']}.png"
        image_bgr = cv2.imread(str(img_path))
        image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

        h, w = image_rgb.shape[:2]
        side = min(h, w)
        top, left = (h - side) // 2, (w - side) // 2
        image_rgb = image_rgb[top:top + side, left:left + side]
        image_rgb = cv2.resize(image_rgb, (self.img_size, self.img_size), interpolation=cv2.INTER_AREA)
        image_rgb = apply_clahe(image_rgb)

        tensor = self.transform(image_rgb)
        label = int(row['diagnosis'])
        return tensor, label

In [ ]:
df = pd.read_csv(DATA_DIR / 'train.csv')
print(df['diagnosis'].value_counts().sort_index())

train_df, val_df = train_test_split(df, test_size=VAL_SPLIT, stratify=df['diagnosis'], random_state=SEED)
print(f'Train: {len(train_df)}  Val: {len(val_df)}')

images_dir = DATA_DIR / 'train_images'
train_ds = APTOSDataset(train_df, images_dir, IMG_SIZE, train=True)
val_ds = APTOSDataset(val_df, images_dir, IMG_SIZE, train=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

## 5. Model — EfficientNet-B4 with a 5-class head

In [ ]:
def build_model():
    # NOTE: timm's EfficientNet-B4, not torchvision's -- different internal
    # layer names, so checkpoints from one won't load into the other.
    # The VisionGuard AI backend expects timm checkpoints.
    return timm.create_model('efficientnet_b4', pretrained=True, num_classes=NUM_CLASSES)

model = build_model().to(device)

# Class weights to counter APTOS's heavy 'No DR' imbalance
class_counts = train_df['diagnosis'].value_counts().sort_index()
weights = torch.tensor((1.0 / class_counts.values), dtype=torch.float32)
weights = (weights / weights.sum() * NUM_CLASSES).to(device)
criterion = nn.CrossEntropyLoss(weight=weights)

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=2, factor=0.5)

## 6. Train

We track **Quadratic Weighted Kappa (QWK)** — the official APTOS competition metric — alongside accuracy, since it penalizes predictions further from the true grade more heavily (confusing Grade 0 with Grade 4 is far worse clinically than confusing Grade 2 with Grade 3).

In [ ]:
def run_epoch(loader, train):
    model.train() if train else model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []

    with torch.set_grad_enabled(train):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            if train:
                optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * images.size(0)
            all_preds.extend(outputs.argmax(dim=1).cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    avg_loss = total_loss / len(loader.dataset)
    qwk = cohen_kappa_score(all_labels, all_preds, weights='quadratic')
    accuracy = float(np.mean(np.array(all_preds) == np.array(all_labels)))
    return avg_loss, accuracy, qwk

In [ ]:
best_qwk = -1.0

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    train_loss, train_acc, train_qwk = run_epoch(train_loader, train=True)
    val_loss, val_acc, val_qwk = run_epoch(val_loader, train=False)
    scheduler.step(val_qwk)

    print(f'Epoch {epoch:02d}/{EPOCHS} ({time.time()-t0:.0f}s) | '
          f'train loss {train_loss:.4f} acc {train_acc:.3f} qwk {train_qwk:.3f} | '
          f'val loss {val_loss:.4f} acc {val_acc:.3f} qwk {val_qwk:.3f}')

    if val_qwk > best_qwk:
        best_qwk = val_qwk
        torch.save({'model_state_dict': model.state_dict(), 'val_qwk': best_qwk, 'epoch': epoch}, OUTPUT_DIR / 'best_model.pth')
        print(f'  ↳ New best QWK {best_qwk:.4f} — checkpoint saved')

torch.save(model.state_dict(), OUTPUT_DIR / 'best_model_last.pth')
print(f'\nTraining complete. Best validation QWK: {best_qwk:.4f}')

## 7. Download the checkpoint

Download `best_model.pth`, then in your VisionGuard AI project:

```bash
cp best_model.pth backend/app/ml/weights/best_model.pth
```

and set `MODEL_MODE=production` in `backend/.env`. Restart the backend — it will automatically load your fine-tuned weights instead of the demo backbone.

In [ ]:
from google.colab import files
files.download(str(OUTPUT_DIR / 'best_model.pth'))